# Purpose

This notebook demonstrates a hierarchical chunking strategy for Retrieval-Augmented Generation (RAG) pipelines. It showcases how to split documents into parent and child chunks, embed only child chunks, find the more relevant child chunks against user query and extract the corresponding parent chunks to be passed to Large Language Models (LLMs).

Since a child chunk is smaller in size, embedding retains it's context better resulting in high-quality matching against user queries. Then, to provide more context around this smaller chunk, corresponding parent chunk is retrieved, which is later appended to user query forming the complete LLM prompt.

- <u>Chunk --> Vector</u>: Since this is for understanding purpose, not using Embedding model to create Vector Embedding for a text chunk. Instead a naive `BoW` (Bag of Words) is used. This is inferior to `TF-IDF` (Term Frequency-Inverse Document Frequency) which is used in Elasticsearch (rather it's optimised version - `BM25`). To map a word to a number (and hence an index for BoW), each word is hased using `MD5`. Note that only child chunks are converted into an embedding (= list of floating-point numbers), they point to corresponding parent chunk.
- <u>Vector Storage</u>: In-memory as hash maps. In production, a Vector DB is used that indices using HNSW (Hierarchical Navigable Small Worlds). Here,
- <u>Query</u>: User query is compared against each of the child chunks as cosine similarity.

In [1]:
import argparse
import uuid
from dataclasses import dataclass, field
from typing import Optional


In [2]:
# ── Chunking helpers ──────────────────────────────────────────────────────────

def split_into_sentences(text: str) -> list[str]:
    """Naive sentence splitter (swap for spaCy/nltk in production)."""
    import re
    # Split on . ! ? followed by whitespace and capital letter
    sentences = re.split(r'(?<=[.!?])\s+(?=[A-Z])', text.strip())
    return [s.strip() for s in sentences if s.strip()]


def token_count(text: str) -> int:
    """Approximate token count (1 token ≈ 4 chars)."""
    return len(text) // 4


In [3]:
def make_parent_chunks(text: str, max_tokens: int = 512) -> list[str]:
    """
    Split text into large parent chunks.
    Accumulates paragraphs as much as it can, respecting `max_tokens`.
    This preserves semantic coherence. Better than small paragraph into its own chunk.
    If a single paragraph exceeds `max_tokens`, it is hard-split until `max_tokens` it hit.
    """
    paragraphs = [p.strip() for p in text.split("\n\n") if p.strip()]
    chunks, chunk = [], ""

    for para in paragraphs:
        # try growing existing chunk
        accumulator = (chunk + "\n\n" + para).strip() if chunk else para
        if token_count(accumulator) <= max_tokens:  # fits
            chunk = accumulator                     # so keep growing
        else:                                       # last addition of `para` exceeded limit
            if chunk:
                chunks.append(chunk)                # add accumulated chunk without last `para`
            # If a single paragraph (last `para`) exceeds limit, hard-split it
            if token_count(para) > max_tokens:
                words = para.split()
                buf = ""
                for word in words:
                    if token_count(buf + " " + word) > max_tokens:
                        chunks.append(buf.strip())
                        buf = word
                    else:
                        buf += " " + word
                chunk = buf.strip()
            else:
                chunk = para

    if chunk:
        chunks.append(chunk)

    return chunks


In [4]:
def make_child_chunks(parent: str, max_tokens: int = 128) -> list[str]:
    """
    Split a parent chunk into small child chunks (sentence groups).
    Each child is short enough to embed precisely.
    """
    sentences = split_into_sentences(parent)
    children, chunk = [], ""

    for sent in sentences:
        accumulator = (chunk + " " + sent).strip() if chunk else sent  # add next sentence
        if token_count(accumulator) <= max_tokens:
            chunk = accumulator   # smaller than limit. Try adding more sentences.
        else:
            if chunk:
                children.append(chunk)  # add the chunk without last sentence.
            chunk = sent

    if chunk:
        children.append(chunk)

    return children



In [5]:
# ── Data model ────────────────────────────────────────────────────────────────

@dataclass
class ChildChunk:
    id: str
    text: str
    parent_id: str
    child_index: int          # position within parent


@dataclass
class ParentChunk:
    id: str
    text: str
    source: str               # filename or doc identifier
    parent_index: int         # position in document
    children: list[ChildChunk] = field(default_factory=list)


In [6]:
def build_hierarchy(text: str, source: str = "doc",
                    parent_max_tokens: int = 512,
                    child_max_tokens: int = 128) -> list[ParentChunk]:
    """
    Full pipeline: text → parent chunks → child chunks, linked together.
    Returns a list of ParentChunk objects each containing their children.
    """
    parent_texts = make_parent_chunks(text, max_tokens=parent_max_tokens)
    parents = []

    for p_idx, p_text in enumerate(parent_texts):
        p_id = str(uuid.uuid4())
        child_texts = make_child_chunks(p_text, max_tokens=child_max_tokens)

        children = [
            ChildChunk(
                id=str(uuid.uuid4()),
                text=c_text,
                parent_id=p_id,
                child_index=c_idx,
            )
            for c_idx, c_text in enumerate(child_texts)
        ]

        parents.append(ParentChunk(
            id=p_id,
            text=p_text,
            source=source,
            parent_index=p_idx,
            children=children,
        ))

    return parents


In [7]:
# ── In-memory vector store (no external DB needed for demo) ───────────────────

class SimpleVectorStore:
    """
    Minimal cosine-similarity store using only numpy.
    In production: swap for ChromaDB / Pinecone / Weaviate.
    """
    def __init__(self):
        self.child_embeddings: list[tuple[ChildChunk, list[float]]] = []
        self.parent_map: dict[str, ParentChunk] = {}

    def _embed(self, text: str) -> list[float]:
        """
        Toy bag-of-words embedding for demo — replace with
        OpenAI text-embedding-3-small or sentence-transformers.
        """
        import hashlib, math
        DIM = 64
        vec = [0.0] * DIM
        for word in text.lower().split():
            idx = int(hashlib.md5(word.encode()).hexdigest(), 16) % DIM
            vec[idx] += 1.0
        norm = math.sqrt(sum(x**2 for x in vec)) or 1.0
        return [x / norm for x in vec]

    def index(self, parents: list[ParentChunk]):
        """Embed all child chunks; store parent lookup."""
        for parent in parents:
            self.parent_map[parent.id] = parent
            for child in parent.children:
                emb = self._embed(child.text)
                self.child_embeddings.append((child, emb))
        print(f"  Indexed {len(self.child_embeddings)} child chunks "
              f"from {len(self.parent_map)} parent chunks.")

    def query(self, question: str, top_k: int = 3) -> list[ParentChunk]:
        """
        1. Embed the query
        2. Find top-k child chunks by cosine similarity   ← small, precise
        3. Return their PARENT chunks to the LLM          ← large, rich context
        """
        import math
        q_emb = self._embed(question)

        def cosine(a, b):
            dot = sum(x*y for x, y in zip(a, b))
            na = math.sqrt(sum(x**2 for x in a)) or 1.0
            nb = math.sqrt(sum(x**2 for x in b)) or 1.0
            return dot / (na * nb)

        scored = [(cosine(q_emb, emb), child)
                  for child, emb in self.child_embeddings]
        scored.sort(key=lambda x: x[0], reverse=True)

        # Deduplicate: multiple children may share a parent
        seen_parents, results = set(), []
        for score, child in scored[:top_k * 3]:
            parent = self.parent_map[child.parent_id]
            if parent.id not in seen_parents:
                seen_parents.add(parent.id)
                results.append((score, child, parent))
            if len(results) == top_k:
                break

        return results



In [8]:
# ── PDF / text loading ────────────────────────────────────────────────────────

def load_pdf(path: str) -> str:
    """Extract plain text from a PDF using pdfplumber."""
    try:
        import pdfplumber
    except ImportError:
        raise ImportError("Run: pip install pdfplumber")

    pages = []
    with pdfplumber.open(path) as pdf:
        for page in pdf.pages:
            text = page.extract_text()
            if text:
                pages.append(text)
    return "\n\n".join(pages)


def load_txt(path: str) -> str:
    with open(path, "r", encoding="utf-8") as f:
        return f.read()



In [9]:
# ── Demo text ─────────────────────────────────────────────────────────────────

DEMO_TEXT = """
Abstract

Retrieval-Augmented Generation (RAG) combines large language models with external
knowledge retrieval to improve factual accuracy and reduce hallucinations. This paper
surveys the key components of RAG pipelines: document parsing, chunking strategies,
embedding models, vector databases, and generation modules.

Introduction

Large language models (LLMs) are powerful but suffer from outdated knowledge and
hallucinations. RAG addresses this by retrieving relevant documents at inference time
and conditioning the LLM on retrieved context. The approach has become widely adopted
in enterprise AI applications.

Chunking Strategies

Chunking is the process of splitting parsed documents into segments suitable for
embedding. Fixed-size chunking splits text every N tokens with optional overlap.
Sentence-based chunking preserves natural language boundaries. Semantic chunking
splits where embedding similarity drops, indicating a topic change.

Hierarchical chunking, also called parent-child or small-to-big retrieval, maintains
two levels of granularity. Small child chunks are embedded for precise retrieval.
Large parent chunks are returned to the LLM for rich contextual generation. This
strategy resolves the tension between retrieval precision and generation quality.

Vector Databases

After chunking, each chunk is embedded into a dense vector. Vector databases such as
Pinecone, Weaviate, Chroma, and FAISS store these vectors and support approximate
nearest-neighbor (ANN) search. HNSW (Hierarchical Navigable Small World) is the most
widely used ANN algorithm, offering sub-linear query time with high recall. Note that
HNSW is an indexing algorithm inside the vector DB, completely separate from the
hierarchical chunking strategy described above.

Conclusion

RAG pipelines require careful design at each stage. Hierarchical chunking is
recommended for long-form documents where both retrieval precision and generation
context matter. Future work includes multimodal RAG, adaptive chunking, and
online index updates.
"""


In [10]:
# ── Main ──────────────────────────────────────────────────────────────────────

def main():
    parser = argparse.ArgumentParser(description="Hierarchical Chunking Demo")
    parser.add_argument("--pdf", help="Path to a PDF file")
    parser.add_argument("--txt", help="Path to a plain text file")
    parser.add_argument("--demo", action="store_true", help="Use built-in demo text")
    parser.add_argument("--parent-tokens", type=int, default=512)
    parser.add_argument("--child-tokens",  type=int, default=128)
    args = parser.parse_args()

    # ── Load text ──
    if args.pdf:
        print(f"\nLoading PDF: {args.pdf}")
        text = load_pdf(args.pdf)
        source = args.pdf
    elif args.txt:
        print(f"\nLoading text file: {args.txt}")
        text = load_txt(args.txt)
        source = args.txt
    else:
        print("\nRunning with built-in demo text (pass --pdf or --txt for your own doc)")
        text = DEMO_TEXT
        source = "demo_rag_whitepaper"

    # ── Build hierarchy ──
    print(f"\nBuilding hierarchy  "
          f"(parent ≤ {args.parent_tokens} tokens, child ≤ {args.child_tokens} tokens)...")
    parents = build_hierarchy(text, source=source,
                              parent_max_tokens=args.parent_tokens,
                              child_max_tokens=args.child_tokens)

    print(f"\n{'─'*60}")
    print(f"  Document  : {source}")
    print(f"  Parents   : {len(parents)}")
    print(f"  Children  : {sum(len(p.children) for p in parents)}")
    print(f"  Avg children/parent: "
          f"{sum(len(p.children) for p in parents)/len(parents):.1f}")
    print(f"{'─'*60}")

    # ── Show structure ──
    print("\nChunk structure preview:\n")
    for p in parents[:3]:  # show first 3 parents
        preview = p.text[:80].replace("\n", " ")
        print(f"  [Parent {p.parent_index}]  ~{token_count(p.text)} tokens")
        print(f"    \"{preview}...\"")
        for c in p.children:
            c_preview = c.text[:60].replace("\n", " ")
            print(f"    └─ [Child {c.child_index}] ~{token_count(c.text)} tokens: "
                  f"\"{c_preview}...\"")
        print()

    # ── Index & query ──
    print("Indexing child chunks into vector store...")
    store = SimpleVectorStore()
    store.index(parents)

    queries = [
        "What is hierarchical chunking?",
        "How does HNSW relate to chunking?",
        "Why is RAG useful for LLMs?",
    ]

    print(f"\n{'─'*60}")
    print("Sample retrieval (child search → parent returned to LLM):")
    print(f"{'─'*60}\n")

    for q in queries:
        print(f"  Query: \"{q}\"")
        results = store.query(q, top_k=1)
        for score, matched_child, parent in results:
            print(f"  Matched child  : \"{matched_child.text[:80]}...\"")
            print(f"  Returned parent: \"{parent.text[:120].replace(chr(10), ' ')}...\"")
            print(f"  Parent tokens  : ~{token_count(parent.text)}")
        print()


# if __name__ == "__main__":
#     main()



In [17]:
main(['--demo'])


Running with built-in demo text (pass --pdf or --txt for your own doc)

Building hierarchy  (parent ≤ 512 tokens, child ≤ 128 tokens)...

────────────────────────────────────────────────────────────
  Document  : demo_rag_whitepaper
  Parents   : 1
  Children  : 5
  Avg children/parent: 5.0
────────────────────────────────────────────────────────────

Chunk structure preview:

  [Parent 0]  ~510 tokens
    "Abstract  Retrieval-Augmented Generation (RAG) combines large language models wi..."
    └─ [Child 0] ~108 tokens: "Abstract  Retrieval-Augmented Generation (RAG) combines larg..."
    └─ [Child 1] ~108 tokens: "RAG addresses this by retrieving relevant documents at infer..."
    └─ [Child 2] ~123 tokens: "Semantic chunking splits where embedding similarity drops, i..."
    └─ [Child 3] ~117 tokens: "Vector databases such as Pinecone, Weaviate, Chroma, and FAI..."
    └─ [Child 4] ~51 tokens: "Hierarchical chunking is recommended for long-form documents..."

Indexing child chunks i